# Trading App V2 — MultiRate 10B Live Shell

This notebook loads the completed 2024 10B MultiRate Transformer predictions, selects the top equity opportunities, refreshes options only for those underlyings, and builds Alpaca paper/live order plans. It does not train Random Forest models, rebuild historical feature panels, or route orders to Robinhood.

In [1]:
from pathlib import Path
import json
import subprocess
import os
import sys
import pandas as pd
import torch
import exchange_calendars as xcals
from dotenv import load_dotenv
from quant_warehouse.warehouse.api import Warehouse
from quant_warehouse.migrate.backfill_missing_fmp import backfill_missing_fmp_historical

REPO_ROOT = Path('/home/jlee153232/PycharmProjects/optimal_trader')
ORCHESTRATOR_ROOT = Path('/home/jlee153232/PycharmProjects/quant-orchestrator')
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(ORCHESTRATOR_ROOT))
load_dotenv(REPO_ROOT / '.env', override=False)
from app.trading_app_v2_runtime import (
    alpaca_client_from_env,
    build_alpaca_equity_orders,
    build_latest_equity_leaderboard,
    build_ranked_alpaca_option_orders,
    build_llm_ranked_option_orders,
    build_score_date_option_ml_ranking_table,
    load_multirate_strategy_scores,
    save_live_artifacts,
    select_optionable_leaderboard,
    write_streamlit_leaderboard_app,
)
print(f'optimal_trader: {REPO_ROOT}')
print(f'quant-orchestrator: {ORCHESTRATOR_ROOT}')

optimal_trader: /home/jlee153232/PycharmProjects/optimal_trader
quant-orchestrator: /home/jlee153232/PycharmProjects/quant-orchestrator


In [2]:
TOP_K = 20
MIN_LONG_SCORE = 0.50
OPTION_STRATEGY_ALLOCATION = 100_000.0
OPTION_TENOR_DAYS = 90
ALPACA_LIVE_OPTION_DISCOUNT_PCT = float(os.getenv('TRADING_APP_V2_ALPACA_LIVE_OPTION_DISCOUNT_PCT', '90.0'))
MODEL_UNIVERSE = os.getenv('TRADING_APP_V2_MODEL_UNIVERSE', '1T')
DATA_UNIVERSE = os.getenv('TRADING_APP_V2_DATA_UNIVERSE', '100B')
SOURCE_CORPUS_PATH = Path(os.getenv('TRADING_APP_V2_MULTIRATE_SOURCE', str(ORCHESTRATOR_ROOT / 'artifacts/multi-rate-mtl/inputs' / DATA_UNIVERSE)))
CHECKPOINT_PATH = Path(os.getenv('TRADING_APP_V2_MULTIRATE_CHECKPOINT', str(ORCHESTRATOR_ROOT / 'artifacts/multi-rate-mtl/1T_option_bucket_smoke_full/multirate_mtl_model.pt' if MODEL_UNIVERSE == '1T' else ORCHESTRATOR_ROOT / 'artifacts/multi-rate-mtl/full_mtl_options_100B_us_issuer_quartiles_vwq_earlystop/multirate_mtl_model.pt' if MODEL_UNIVERSE == '100B' else ORCHESTRATOR_ROOT / 'artifacts/multi-rate-mtl/inference_10B_2024_model/multirate_mtl_model.pt')))
LIVE_DIR = REPO_ROOT / 'artifacts/trading_app_v2/live'
FEATURES_PATH = LIVE_DIR / f'multirate_eod_features_{DATA_UNIVERSE}'
OPTION_RANKER_DIR = REPO_ROOT / 'artifacts/trading_app_v2/option_family_ranker'
if not SOURCE_CORPUS_PATH.exists() or not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'MultiRate source/checkpoint not found: {SOURCE_CORPUS_PATH} / {CHECKPOINT_PATH}')
CUDA_AVAILABLE = torch.cuda.is_available()
INFERENCE_PYTHON = sys.executable
INFERENCE_DEVICE = 'cuda' if CUDA_AVAILABLE else 'cpu'
nyse = xcals.get_calendar('XNYS')
eod_today = pd.Timestamp.now(tz='America/Los_Angeles').normalize().tz_localize(None)
sessions = nyse.sessions_in_range((eod_today - pd.Timedelta(days=14)).date(), eod_today.date())
completed_sessions = sessions[sessions < eod_today]
if len(completed_sessions) == 0:
    raise RuntimeError('No prior completed NYSE session is available.')
score_date = completed_sessions[-1].strftime('%Y-%m-%d')
display({'model_universe': MODEL_UNIVERSE, 'data_universe': DATA_UNIVERSE, 'fresh_features': str(FEATURES_PATH), 'checkpoint': str(CHECKPOINT_PATH), 'top_k': TOP_K, 'score_date': score_date, 'score_date_policy': 'one prior completed NYSE EOD date for every symbol', 'inference_device': INFERENCE_DEVICE, 'live_option_discount_pct': ALPACA_LIVE_OPTION_DISCOUNT_PCT})

{'model_universe': '1T',
 'data_universe': '100B',
 'fresh_features': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2/live/multirate_eod_features_100B',
 'checkpoint': '/home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multi-rate-mtl/1T_option_bucket_smoke_full/multirate_mtl_model.pt',
 'top_k': 20,
 'score_date': '2026-09-08',
 'score_date_policy': 'one prior completed NYSE EOD date for every symbol',
 'inference_device': 'cuda',
 'live_option_discount_pct': 90.0}

In [3]:
LIVE_DIR.mkdir(parents=True, exist_ok=True)
warehouse = Warehouse()
source_taxonomy = pd.read_csv(SOURCE_CORPUS_PATH / 'taxonomy.csv')
equity_symbols = sorted({str(symbol).strip().upper() for symbol in source_taxonomy['symbol'] if str(symbol).strip() and not (len(str(symbol).strip()) == 5 and str(symbol).strip().upper().endswith('X'))})
from quant_warehouse.platforms.data_providers.thetadata.options import (OPTIONS_THETADATA_EOD_LIBRARY, OPTIONS_THETADATA_PROVIDER, option_chain_storage_symbol)
from quant_warehouse.warehouse.storage import provider_library
option_library = provider_library(OPTIONS_THETADATA_EOD_LIBRARY, OPTIONS_THETADATA_PROVIDER)
stored_option_symbols = set(warehouse.backend.list_symbols(option_library))
optionable_symbols = sorted(symbol for symbol in equity_symbols if option_chain_storage_symbol(symbol) in stored_option_symbols)
print(f'[option-universe] local ThetaData symbols with any stored historical option data: {len(optionable_symbols)}', flush=True)
optionable_symbols = sorted(optionable_symbols)
if not optionable_symbols:
    raise RuntimeError(f'No locally downloaded ThetaData option history found from 2022-01-01 through {score_date}. Download the historical EOD data first.')
OPTIONABLE_SYMBOLS_PATH = LIVE_DIR / 'optionable_symbols.csv'
pd.DataFrame({'symbol': optionable_symbols}).to_csv(OPTIONABLE_SYMBOLS_PATH, index=False)
print(f'Refreshing FMP/Quant Warehouse data for {len(optionable_symbols)} optionable $10B+ symbols through {score_date}...', flush=True)
refresh_summary = backfill_missing_fmp_historical(warehouse=warehouse, equity_symbols=optionable_symbols, etf_symbols=(), include_macro=False, include_prices=True, staleness_days=0, skip_recent_hours=0, max_workers=8, progress_logger=print)
display({'fmp_refresh_status': refresh_summary.get('status'), 'refreshed_symbols': len(refresh_summary.get('equity_symbols', []))})
# N-PORT is not required for this live equity/option scoring path; skip the 4,425-fund backfill.
display({'fund_nport_status': 'skipped_for_live_scoring'})
expected_feature_families = set(json.loads((SOURCE_CORPUS_PATH / 'manifest.json').read_text()).get('feature_families', []))
existing_feature_families = set(json.loads((FEATURES_PATH / 'manifest.json').read_text()).get('feature_families', [])) if (FEATURES_PATH / 'manifest.json').exists() else set()
if not (FEATURES_PATH / 'manifest.json').exists() or existing_feature_families != expected_feature_families:
    feature_build_cmd = [INFERENCE_PYTHON, str(ORCHESTRATOR_ROOT / 'scripts/build_multirate_mtl_corpus.py'), '--symbols', str(OPTIONABLE_SYMBOLS_PATH), '--target-events', str(SOURCE_CORPUS_PATH / 'target_events.parquet'), '--output-dir', str(FEATURES_PATH), '--chunk-size', '100', '--start-date', '1900-01-01', '--text-device', INFERENCE_DEVICE]
    print('Building fresh MultiRate feature inputs from Quant Warehouse...', flush=True)
    subprocess.run(feature_build_cmd, cwd=ORCHESTRATOR_ROOT, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
if (SOURCE_CORPUS_PATH / 'taxonomy.csv').exists():
    import shutil
    shutil.copy2(SOURCE_CORPUS_PATH / 'taxonomy.csv', FEATURES_PATH / 'taxonomy.csv')
inference_cmd = [INFERENCE_PYTHON, str(ORCHESTRATOR_ROOT / 'scripts/train_multirate_mtl.py'), '--corpus', str(FEATURES_PATH), '--output-dir', str(LIVE_DIR / 'multirate_inference'), '--checkpoint', str(CHECKPOINT_PATH), '--inference-only', '--prediction-start-date', score_date, '--option-panel', str(ORCHESTRATOR_ROOT / 'artifacts/multi-rate-mtl/inputs/annual_option_dte_documents_2025_v5/selected_contract_documents.parquet'), '--option-start-date', '2025-01-01', '--skip-embeddings', '--skip-t-sne', '--device', INFERENCE_DEVICE]
print('Starting MultiRate inference; progress will be reported for each scoring batch...', flush=True)
subprocess.run(inference_cmd, cwd=ORCHESTRATOR_ROOT, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
generated_predictions = LIVE_DIR / 'multirate_inference' / 'supervised_predictions.csv'
if not generated_predictions.exists():
    raise FileNotFoundError(f'MultiRate inference did not create {generated_predictions}')
strategy_scores = load_multirate_strategy_scores(generated_predictions)
eod_strategy_scores = strategy_scores.loc[pd.to_datetime(strategy_scores['date'], errors='coerce').eq(pd.Timestamp(score_date))].copy()
if eod_strategy_scores.empty:
    raise RuntimeError(f'No MultiRate scores were produced for required EOD date {score_date}. Refresh the corpus features through that date.')
leaderboard = build_latest_equity_leaderboard(eod_strategy_scores, top_k=TOP_K, min_long_score=MIN_LONG_SCORE, price_provider='fmp')
display(leaderboard.head(TOP_K + 5))
print({'scored_equity_symbols': int(eod_strategy_scores['symbol'].nunique()), 'selected_equities': int(leaderboard['selected'].sum())})

[option-universe] local ThetaData symbols with any stored historical option data: 119
Refreshing FMP/Quant Warehouse data for 119 optionable $10B+ symbols through 2026-09-08...
Backfill: scoped equity symbols (119): AAPL, ABBV, ABT, ADI, AMAT, AMD, AMGN, AMZN, ANET, ANTM, APH, APP, ASML, AVGO, AXP, BA, BAC, BKNG, BLK, BMY, BNY, BX, C, CAT, CDNS, COF, COP, COST, CRM, CRWD, CSCO, CVS, CVX, DE, DELL, DHR, DIS, DUK, ETN, FTNT, GD, GE, GEV, GILD, GLW, GOOG, GOOGL, GS, HD, HOOD, HWM, IBKR, IBM, INTC, ISRG, JNJ, JPM, KLAC, KO, LLY, LMT, LOW, LRCX, MA, MCD, MDT, META, MO, MRK, MRVL, MS, MSFT, MU, NEE, NEM, NFLX, NOW, NVDA, ORCL, PANW, PEP, PFE, PG, PGR, PH, PLD, PLTR, PM, PNC, PWR, QCOM, RTX, SBUX, SCCO, SCHW, SNDK, SO, SPCX, SPGI, SYK, ... (+19 more)
Backfill: refreshing equity prices for 1 stale symbols (119 scoped) via FMP
Warehouse price refresh progress: 1/1 symbols processed
Backfill: refreshing equity fundamentals for 119 stale symbols (119 scoped) | sections=income,balance,cash,metrics

{'fmp_refresh_status': None, 'refreshed_symbols': 119}

{'fund_nport_status': 'skipped_for_live_scoring'}

Starting MultiRate inference; progress will be reported for each scoring batch...
[multirate-inference] exact-date evaluation set: 118 symbols on 2026-09-08


/home/jlee153232/PycharmProjects/quant-orchestrator/quant_orchestrator/platforms/ml_frameworks/torch/models/transformers/multirate/model.py:254: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  return nn.TransformerEncoder(layer, num_layers=config.layers)


[multirate-inference] scoring symbols 1-32/118: AAPL,ABBV,ABT,ADI,AMAT,AMD,AMGN,AMZN,ANET,APH,APP,ASML,AVGO,AXP,BA,BAC,BKNG,BLK,BMY,BNY,BX,C,CAT,CDNS,COF,COP,COST,CRM,CRWD,CSCO,CVS,CVX
[multirate-inference] scoring symbols 33-64/118: DE,DELL,DHR,DIS,DUK,ETN,FTNT,GD,GE,GEV,GILD,GLW,GOOG,GOOGL,GS,HD,HOOD,HWM,IBKR,IBM,INTC,ISRG,JNJ,JPM,KLAC,KO,LLY,LMT,LOW,LRCX,MA,MCD
[multirate-inference] scoring symbols 65-96/118: MDT,META,MO,MRK,MRVL,MS,MSFT,MU,NEE,NEM,NFLX,NOW,NVDA,ORCL,PANW,PEP,PFE,PG,PGR,PH,PLD,PLTR,PM,PNC,PWR,QCOM,RTX,SBUX,SCCO,SCHW,SNDK,SO
[multirate-inference] scoring symbols 97-118/118: SPCX,SPGI,SYK,T,TJX,TMO,TMUS,TSLA,TT,TXN,UBER,UNH,UNP,V,VRT,VRTX,VZ,WDC,WELL,WFC,WMT,XOM


,symbol,score_date,prob_buy,prob_short,model_count,best_family_score,direction,confidence,rank,close,eligible,capacity_rank,selected
0,SPCX,2026-09-08,0.527371,0.514311,1,0.527371,long,0.527371,1,147.55,True,1,True
1,AMZN,2026-09-08,0.524574,0.500975,1,0.524574,long,0.524574,2,252.40,True,2,True
2,META,2026-09-08,0.523692,0.501452,1,0.523692,long,0.523692,3,653.69,True,3,True
3,WMT,2026-09-08,0.523122,0.496416,1,0.523122,long,0.523122,4,105.83,True,4,True
4,NVDA,2026-09-08,0.521635,0.500032,1,0.521635,long,0.521635,5,223.67,True,5,True
5,BNY,2026-09-08,0.520894,0.510894,1,0.520894,long,0.520894,6,162.51,True,6,True
6,BA,2026-09-08,0.520345,0.515031,1,0.520345,long,0.520345,7,206.42,True,7,True
7,TSLA,2026-09-08,0.519729,0.503250,1,0.519729,long,0.519729,8,367.81,True,8,True
8,LLY,2026-09-08,0.519452,0.500012,1,0.519452,long,0.519452,9,1124.21,True,9,True
9,GOOGL,2026-09-08,0.519360,0.498351,1,0.519360,long,0.519360,10,330.65,True,10,True


{'scored_equity_symbols': 118, 'selected_equities': 20}


In [4]:
score_date = None
print({'score_date_policy': 'latest completed prediction date per symbol', 'score_date_min': leaderboard['score_date'].min(), 'score_date_max': leaderboard['score_date'].max()})
option_leaderboard = select_optionable_leaderboard(leaderboard, score_date=score_date, top_k=TOP_K, option_data_source='thetadata')
selected_symbols = option_leaderboard['symbol'].astype(str).str.upper().tolist()
print({'score_date': score_date, 'option_underlyings': selected_symbols})
print('Using locally downloaded ThetaData for EOD option features; Alpaca is reserved for live quotes and orders.')

{'score_date_policy': 'latest completed prediction date per symbol', 'score_date_min': Timestamp('2026-09-08 00:00:00'), 'score_date_max': Timestamp('2026-09-08 00:00:00')}
{'score_date': None, 'option_underlyings': ['AMZN', 'WMT', 'BNY', 'BA', 'AAPL', 'GOOG', 'DUK', 'VZ', 'DIS', 'HD', 'BKNG', 'DHR', 'BX', 'TMUS', 'AVGO', 'GILD', 'GD', 'BLK', 'CVX', 'WFC']}
Using locally downloaded ThetaData for EOD option features; Alpaca is reserved for live quotes and orders.


In [5]:
option_rankings = build_score_date_option_ml_ranking_table(
    OPTION_RANKER_DIR, leaderboard=option_leaderboard, score_date=score_date,
    symbols=selected_symbols, target_dte=OPTION_TENOR_DAYS, min_market_cap=10_000_000_000,
    start_date='1900-01-01', max_underlyings=TOP_K, option_data_source='thetadata',
)
display(option_rankings)
print(f'option ranking rows: {len(option_rankings)}')

,selected_by_option_ensemble,option_ensemble_rank,option_ensemble_mean_score,option_family_score_count,trade_id,symbol,side,equity_signal_side,entry_date,option_type,...,abs_gamma,abs_theta,abs_vega,abs_rho,theta_to_mid,vega_to_mid,iv_expiration_z,iv_times_sqrt_dte,fallback_option_score,option_score_source
0,True,1,1843.146215,1,live|AAPL|2026-09-08|call,AAPL,long,long,2026-09-08,call,...,0.0451,1.4450,5.9552,0.2767,-0.862687,3.555343,-0.541249,0.025224,1843.146215,local_thetadata_fallback
1,True,1,1562.279740,1,live|AMZN|2026-09-08|call,AMZN,long,long,2026-09-08,call,...,0.0926,0.8606,5.3339,0.3173,-0.583458,3.616203,-0.636389,0.016661,1562.279740,local_thetadata_fallback
2,True,1,884.865489,1,live|AVGO|2026-09-08|call,AVGO,long,long,2026-09-08,call,...,0.0473,1.6546,7.6017,0.4345,-0.619700,2.847079,-0.307962,0.022565,884.865489,local_thetadata_fallback
3,True,1,194.600949,1,live|BA|2026-09-08|call,BA,long,long,2026-09-08,call,...,0.0252,0.2035,3.3375,0.1694,-0.626154,10.269231,-0.927350,0.032846,194.600949,local_thetadata_fallback
4,True,1,89.520284,1,live|BKNG|2026-09-08|call,BKNG,long,long,2026-09-08,call,...,0.0111,0.0859,29.1571,10.5754,-0.013422,4.555797,-0.410867,0.180630,89.520284,local_thetadata_fallback
5,True,1,-10.933976,1,live|BLK|2026-09-08|call,BLK,long,long,2026-09-08,call,...,0.0025,0.3563,229.8485,134.3319,-0.006357,4.100776,-0.939251,0.142292,-10.933976,local_thetadata_fallback
6,True,1,-8.742485,1,live|BNY|2026-09-08|call,BNY,long,long,2026-09-08,call,...,0.0023,0.0253,7.2198,29.5837,-0.000510,0.145560,1.626538,0.216253,-8.742485,local_thetadata_fallback
7,True,1,66.225859,1,live|BX|2026-09-08|put,BX,short,short,2026-09-08,put,...,0.0398,0.1504,7.7860,1.1746,-0.083788,4.337604,-0.253080,0.065778,66.225859,local_thetadata_fallback
8,True,1,4.257463,1,live|CVX|2026-09-08|put,CVX,short,short,2026-09-08,put,...,0.0001,0.0031,0.9132,0.1753,-0.044286,13.045714,1.192763,0.375168,4.257463,local_thetadata_fallback
9,True,1,-0.222982,1,live|DHR|2026-09-08|call,DHR,long,long,2026-09-08,call,...,0.0102,0.0752,39.0806,32.3542,-0.003642,1.892523,-0.914431,0.172697,-0.222982,local_thetadata_fallback


option ranking rows: 20


In [6]:
paper_order_plans = {}
paper_order_clients = {}
paper_order_plans['alpaca_equity_paper'] = build_alpaca_equity_orders(leaderboard=leaderboard, account_prefix='EQUITY', gross_exposure=0.95)
paper_order_plans['alpaca_option_paper'] = build_ranked_alpaca_option_orders(option_rankings=option_rankings, decisions=option_leaderboard[['symbol', 'direction']], account_prefix='OPTION', strategy_allocation=OPTION_STRATEGY_ALLOCATION, live=False)
paper_order_plans['alpaca_llm_paper'], llm_reviews = build_llm_ranked_option_orders(leaderboard=option_leaderboard, option_rankings=option_rankings, top_k=TOP_K, account_prefix='LLM', strategy_allocation=OPTION_STRATEGY_ALLOCATION)
paper_order_plans['alpaca_option_live'] = build_ranked_alpaca_option_orders(option_rankings=option_rankings, decisions=option_leaderboard[['symbol', 'direction']], account_prefix='OPTION', strategy_allocation=OPTION_STRATEGY_ALLOCATION, live=True, discount_pct=ALPACA_LIVE_OPTION_DISCOUNT_PCT)
for name, frame in paper_order_plans.items():
    print(name, len(frame))
    display(frame.head(20))

alpaca_equity_paper 20


,symbol,action,side,qty,order_type,time_in_force
0,SPCX,open_long,buy,32,market,day
1,AMZN,open_long,buy,18,market,day
2,META,open_long,buy,7,market,day
3,WMT,open_long,buy,44,market,day
4,NVDA,open_long,buy,21,market,day
5,BNY,open_long,buy,29,market,day
6,BA,open_long,buy,23,market,day
7,TSLA,open_long,buy,12,market,day
8,LLY,open_long,buy,4,market,day
9,GOOGL,open_long,buy,14,market,day


alpaca_option_paper 14


,symbol,underlying_symbol,option_type,action,side,qty,bid_price,ask_price,skip_submit,skip_reason,order_type,time_in_force,limit_price,limit_order_price,price,limit_price_source,live_quote_priced_at,discount_pct
0,AMZN260909C00257500,AMZN,call,buy_to_open_call,buy,1666,0.00,0.03,True,missing_bid_price,limit,gtc,NaN,NaN,NaN,NaN,NaN,NaN
1,WMT260918C00110000,WMT,call,buy_to_open_call,buy,135,0.28,0.37,False,,limit,gtc,0.28,0.28,0.28,bid_price,2026-09-10T00:06:24.275051+00:00,0.0
2,BA260911C00220000,BA,call,buy_to_open_call,buy,625,0.05,0.08,False,,limit,gtc,0.05,0.05,0.05,bid_price,2026-09-10T00:06:24.275051+00:00,0.0
3,AAPL260909C00320000,AAPL,call,buy_to_open_call,buy,1666,0.00,0.03,True,missing_bid_price,limit,gtc,NaN,NaN,NaN,NaN,NaN,NaN
4,GOOG260911C00350000,GOOG,call,buy_to_open_call,buy,227,0.12,0.22,False,,limit,gtc,0.12,0.12,0.12,bid_price,2026-09-10T00:06:24.275051+00:00,0.0
5,DUK261218C00125000,DUK,call,buy_to_open_call,buy,16,2.62,3.12,False,,limit,gtc,2.62,2.62,2.62,bid_price,2026-09-10T00:06:24.275051+00:00,0.0
6,DIS261120C00110000,DIS,call,buy_to_open_call,buy,13,3.37,3.66,False,,limit,gtc,3.35,3.35,3.35,bid_price,2026-09-10T00:06:24.275051+00:00,0.0
7,BKNG261120C00200000,BKNG,call,buy_to_open_call,buy,11,3.98,4.45,False,,limit,gtc,3.95,3.95,3.95,bid_price,2026-09-10T00:06:24.275051+00:00,0.0
8,DHR261218C00195000,DHR,call,buy_to_open_call,buy,2,19.07,21.47,False,,limit,gtc,19.05,19.05,19.05,bid_price,2026-09-10T00:06:24.275051+00:00,0.0
9,TMUS261218C00200000,TMUS,call,buy_to_open_call,buy,9,4.14,5.04,False,,limit,gtc,4.10,4.10,4.10,bid_price,2026-09-10T00:06:24.275051+00:00,0.0


alpaca_llm_paper 0


""


alpaca_option_live 13


,symbol,underlying_symbol,option_type,action,side,qty,bid_price,ask_price,skip_submit,skip_reason,order_type,time_in_force,limit_price,limit_order_price,price,limit_price_source,live_quote_priced_at,discount_pct
0,AMZN260909C00257500,AMZN,call,buy_to_open_call,buy,1329,0.00,0.03,True,missing_bid_price,limit,gtc,NaN,NaN,NaN,NaN,NaN,NaN
1,WMT260918C00110000,WMT,call,buy_to_open_call,buy,107,0.28,0.37,False,,limit,gtc,0.02,0.02,0.02,bid_price,2026-09-10T00:07:12.976043+00:00,90.0
2,BA260911C00220000,BA,call,buy_to_open_call,buy,498,0.05,0.08,True,invalid_bid_price,limit,gtc,NaN,NaN,NaN,NaN,NaN,NaN
3,AAPL260909C00320000,AAPL,call,buy_to_open_call,buy,1329,0.00,0.03,True,missing_bid_price,limit,gtc,NaN,NaN,NaN,NaN,NaN,NaN
4,GOOG260911C00350000,GOOG,call,buy_to_open_call,buy,181,0.12,0.22,False,,limit,gtc,0.01,0.01,0.01,bid_price,2026-09-10T00:07:12.976043+00:00,90.0
5,DUK261218C00125000,DUK,call,buy_to_open_call,buy,12,2.62,3.12,False,,limit,gtc,0.26,0.26,0.26,bid_price,2026-09-10T00:07:12.976043+00:00,90.0
6,DIS261120C00110000,DIS,call,buy_to_open_call,buy,10,3.37,3.66,False,,limit,gtc,0.33,0.33,0.33,bid_price,2026-09-10T00:07:12.976043+00:00,90.0
7,BKNG261120C00200000,BKNG,call,buy_to_open_call,buy,8,3.98,4.45,False,,limit,gtc,0.39,0.39,0.39,bid_price,2026-09-10T00:07:12.976043+00:00,90.0
8,DHR261218C00195000,DHR,call,buy_to_open_call,buy,1,19.07,21.47,False,,limit,gtc,1.90,1.90,1.90,bid_price,2026-09-10T00:07:12.976043+00:00,90.0
9,TMUS261218C00200000,TMUS,call,buy_to_open_call,buy,7,4.14,5.04,False,,limit,gtc,0.41,0.41,0.41,bid_price,2026-09-10T00:07:12.976043+00:00,90.0


In [7]:
symbol_scores = strategy_scores.copy()
saved = save_live_artifacts(live_dir=LIVE_DIR, leaderboard=leaderboard, symbol_scores=symbol_scores, option_ml_rankings=option_rankings, orders=paper_order_plans)
streamlit_app = write_streamlit_leaderboard_app(live_dir=LIVE_DIR, leaderboard=leaderboard, symbol_scores=symbol_scores, option_ml_rankings=option_rankings, orders=paper_order_plans)
print({'saved_live_artifacts': {str(k): str(v) for k, v in saved.items()}, 'streamlit_app': str(streamlit_app)})
import socket
def first_free_streamlit_port(start=8501, stop=8600):
    for port in range(start, stop):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
            if probe.connect_ex(('127.0.0.1', port)) != 0:
                return port
    return start
streamlit_port = first_free_streamlit_port()
print(f'Run: streamlit run {streamlit_app} --server.address 127.0.0.1 --server.port {streamlit_port}')
print(f'Streamlit URL: http://localhost:{streamlit_port}')
print(f'Streamlit URL: http://127.0.0.1:{streamlit_port}')
print('Review the generated Alpaca paper/live plans before using the Streamlit submit button.')

{'saved_live_artifacts': {'leaderboard': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2/live/leaderboard_latest.csv', 'symbol_scores': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2/live/symbol_scores.csv', 'option_ml_rankings': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2/live/option_ml_rankings.csv', 'metadata': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2/live/metadata.json', 'alpaca_equity_paper_orders': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2/live/alpaca_equity_paper_orders.csv', 'alpaca_option_paper_orders': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2/live/alpaca_option_paper_orders.csv', 'alpaca_llm_paper_orders': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2/live/alpaca_llm_paper_orders.csv', 'alpaca_option_live_orders': '/home/jlee153232/PycharmProjects/optimal_trader/artifact